# MOM6 layer-vortdiag-obdy: daily native and remapped diagnostics

This notebook selects this experiment's archived output. Dynamics use **10 native
isopycnal layers**; MOM6 remaps supported diagnostics onto **29 z-star cells**.
Their nominal centres include the 28 requested depths and 3734.55 m. These are
cell averages on a weakly moving z-star grid, treated as fixed depths here.
Native RV/PV and the primary vorticity budget remain layer-based.
All evolving diagnostics are daily means. Horizontal coordinates are metres.


In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import yaml

## Select and verify the output

**For future runs:** restart the kernel and run all cells. With `experiment_name = None`, the notebook reads the current experiment name from `metadata.yaml` and the laboratory location from the main `config.yaml`. With `segment = None`, it selects the latest available complete archived segment of that experiment. Fresh experiments normally begin at `output000`; continuations add `output001`, etc.

To inspect a specific segment, set `segment = 'output001'` below. To inspect an older experiment, set `experiment_name` to its archive directory name. Set either option back to `None` to restore automatic selection.

If the current experiment has no complete archived diagnostics, the notebook stops and reports any fatal error in its archived logs. It never substitutes an older experiment. Bathymetry and grid metrics are read from the selected output; forcing is read from the current configuration directory.

In [ ]:
root = Path('/g/data/fu5/users/rd7475/github/phd/idealised_modelling/mom6_sandbox/mom6-eddy-exp1-layer-vortdiag-obdy')

# Leave these as None for the current experiment and its latest available segment.
experiment_name = None  # Override only to inspect another archive explicitly.
segment = None          # Example override: 'output000' or 'output001'

metadata = yaml.safe_load((root / 'metadata.yaml').read_text())
run_config = yaml.safe_load((root / 'config.yaml').read_text())
if experiment_name is None:
    experiment_name = metadata['name']
laboratory = Path(run_config['laboratory'])
if not laboratory.is_absolute():
    laboratory = root / laboratory
archive = laboratory / 'archive' / experiment_name
required = ['ocean_static.nc', 'ocean_daily.nc', 'ocean.stats.nc',
            'ocean_layer_daily.nc', 'ocean_budget_daily.nc', 'ocean_budget_z_daily.nc']

def complete_segment(folder):
    if not all((folder / name).is_file() for name in required):
        return False
    with xr.open_dataset(folder / 'ocean_daily.nc', decode_times=False) as probe:
        if not {'thetao', 'uo', 'vo'}.issubset(probe):
            return False
    exitcode = folder / 'exitcode'
    return not exitcode.exists() or exitcode.read_text().strip() == '0'

available = sorted(
    (folder for folder in archive.glob('output*')
     if folder.is_dir() and folder.name[6:].isdigit() and complete_segment(folder)),
    key=lambda folder: int(folder.name[6:]),
)
print('Selected experiment:', experiment_name)
print('Available complete segments:', [folder.name for folder in available])
if segment is None:
    output = available[-1] if available else None
else:
    if not segment.startswith('output') or not segment[6:].isdigit():
        raise ValueError("segment must be None or a name such as 'output000'.")
    output = archive / segment
if output is None or not complete_segment(output):
    log_folder = archive / 'error_logs'
    fatal = None
    for log in sorted(log_folder.glob('*.err'), key=lambda path: path.stat().st_mtime, reverse=True):
        fatal = next((line.strip() for line in log.read_text(errors='replace').splitlines()
                      if 'FATAL' in line), None)
        if fatal:
            break
    detail = f' Latest recorded error: {fatal}' if fatal else ''
    raise FileNotFoundError(
        f'No complete diagnostics for {experiment_name}'
        + (f'/{segment}' if segment else '')
        + f' in {archive}. Check job status and {log_folder}.' + detail
        + ' No older experiment has been substituted.'
    )
time_decoder = xr.coders.CFDatetimeCoder(use_cftime=True)

static = xr.open_dataset(output / 'ocean_static.nc', decode_times=time_decoder)
daily = xr.open_dataset(output / 'ocean_daily.nc', decode_times=time_decoder)
vertical = daily
native = xr.open_dataset(output / 'ocean_layer_daily.nc', decode_times=time_decoder)
budget_native = xr.open_dataset(output / 'ocean_budget_daily.nc', decode_times=time_decoder)
budget_z = xr.open_dataset(output / 'ocean_budget_z_daily.nc', decode_times=time_decoder)
# SSH is two-dimensional and comes from the native module. The first run
# omitted the unsupported ocean_model_z/zos request; use its native SSH.
if 'zos' not in daily:
    daily['zos'] = native.zos
temperature_source = 'ocean_daily.nc (online z-star remapping)'
stats = xr.open_dataset(output / 'ocean.stats.nc', decode_times=time_decoder)
forcing = xr.open_dataset(root / 'input' / 'forcing.nc')
datasets = [static, daily, stats, forcing, native, budget_native, budget_z]
if vertical is not daily:
    datasets.append(vertical)

for ds in [static, daily, vertical]:
    for name in ['xh', 'yh']:
        if ds[name].attrs.get('units') not in ('m', 'meter', 'meters', 'metre', 'metres'):
            raise ValueError(f'{output.name}: {name} is not in metres; select a metre-grid run.')
        ds[name].attrs.update(long_name='x position' if name == 'xh' else 'y position', units='m')
for name in ['x', 'y']:
    forcing[name].attrs.update(long_name=f'{name} position', units='m')
np.testing.assert_allclose(forcing.x.values, daily.xh.values)
np.testing.assert_allclose(forcing.y.values, daily.yh.values)
wet = static.wet.squeeze(drop=True) > 0
x_index = daily.sizes['xh'] // 2
x_section = float(daily.xh.isel(xh=x_index))
print('Output:', output)
print('Daily records:', daily.sizes['time'], '| first:', daily.time.values[0], '| last:', daily.time.values[-1])
print('Cell-centre x range (m):', float(daily.xh.min()), float(daily.xh.max()))
print('Cell-centre y range (m):', float(daily.yh.min()), float(daily.yh.max()))
for name in ['dxt', 'dyt']:
    print(f'{name} range (m):', float(static[name].min()), float(static[name].max()))
print('Section x (m):', x_section)
print('Temperature source:', temperature_source)
print('Last temperature averaging bounds:', vertical.time_bnds.isel(time=-1).values)

print('Native layers:', native.sizes['zl'])
print('Remapped cells:', daily.sizes['z_l'])
print('Nominal remapped centres (m):', daily.z_l.values)
print('Nominal remapped interfaces (m):', daily.z_i.values)
assert daily.sizes['z_l'] == 29, 'Selected run does not use the agreed 29-cell grid.'
np.testing.assert_allclose(daily.z_i.values[[0, -1]], [0, 4000])


## Bathymetry and wind forcing

Bathymetry comes from this run. Both wind-stress components are shown along the selected x position. SST restoring is disabled in the current configuration, so no target-SST plot is included.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
static.deptho.where(wet).plot(ax=axes[0], x='xh', y='yh', cmap='Blues', cbar_kwargs={'label': 'Depth (m)'})
axes[0].set_title('Model bathymetry')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
for ax, name in zip(axes[1:], ['tau_x', 'tau_y']):
    forcing[name].sel(x=x_section, method='nearest').plot(ax=ax, y='y')
    ax.set_title(f'{name} at x = {x_section:,.0f} m')
    ax.set_xlabel('Wind stress (Pa)')
    ax.set_ylabel('y (m)')
plt.tight_layout()

## Uppermost remapped temperature and sea-surface height

Last daily means. Temperature is the uppermost remapped cell average, centred
nominally at 1.78 m; SSH is a two-dimensional native diagnostic.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name, title, cmap in zip(axes, ['thetao', 'zos'], ['Uppermost remapped temperature', 'Sea-surface height'], ['RdYlBu_r', 'RdBu_r']):
    field = daily[name].isel(time=-1)
    if 'z_l' in field.dims:
        field = field.isel(z_l=0)
    field.where(wet).plot(ax=ax, x='xh', y='yh', cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_aspect('equal')
print('Daily averaging bounds:', daily.time_bnds.isel(time=-1).values)
plt.tight_layout()

## Daily temperature on the 29-cell z-star grid

Sections and the uppermost-cell map read online-remapped `thetao` from
`ocean_daily.nc`. Depth axes use nominal z-star cell centres; they are distinct
from the moving native isopycnal interfaces. Depth increases downward.


In [ ]:
section = vertical.thetao.isel(time=-1, xh=x_index).where(wet.isel(xh=x_index))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, title in zip(axes, ['Full depth', 'Upper 200 m']):
    section.plot(ax=ax, x='yh', y='z_l', yincrease=False, cmap='RdYlBu_r')
    ax.set_xlabel('y (m)')
    ax.set_ylabel('Depth (m)')
    ax.set_title(f'{title}: x = {x_section:,.0f} m')
axes[1].set_ylim(200, 0)
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 6))
surface = vertical.thetao.isel(time=-1, z_l=0).where(wet)
surface.plot(ax=ax, x='xh', y='yh', cmap='RdYlBu_r')
ax.set_title(f'Temperature at depth = {float(surface.z_l):g} m')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_aspect('equal')
plt.tight_layout()

## Basic model checks

Energy summed over layers and maximum transport CFL for this segment. These diagnostics describe transient evolution; they do not establish equilibrium.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
stats.KE.sum('Layer', keep_attrs=True).plot(ax=axes[0], marker='o')
axes[0].set_title('Kinetic energy summed over layers')
stats.max_CFL_trans.plot(ax=axes[1], marker='o')
axes[1].set_title('Maximum transport CFL')
plt.tight_layout()
for name in ['thetao', 'zos']:
    field = daily[name].isel(time=-1)
    if 'z_l' in field.dims:
        field = field.isel(z_l=0)
    field = field.where(wet)
    print(name, 'min:', float(field.min()), 'max:', float(field.max()), field.attrs.get('units', ''))
print('Maximum transport CFL in segment:', float(stats.max_CFL_trans.max()))

## Eddy velocity animation: surface, zonal cut, meridional cut

Panels are ordered **(1) surface speed, (2) meridional velocity v along the zonal cut, (3) zonal velocity u along the meridional cut**. The cuts stay at the physical domain midpoint (x = y = 250,000 m for this grid); both sections show 0–4,000 m depth. Positive v is northward and positive u is eastward.

The surface panel uses the uppermost depth level. Staggered u/v are interpolated to tracer centres before computing their norm. Because the diagnostics are daily means, the plotted quantity is the **speed of the daily mean velocity**, rather than the mean of instantaneous speed. Bathymetry and wet masks hide land and below-bottom cells.

Colour scales remain fixed across all frames; the two signed sections share a symmetric scale. All panels use the same daily records. The HTML player provides play/pause, a frame slider and playback controls. Change `frame_stride` to skip records or `interval_ms` to adjust playback speed. Only the surface and two sections are loaded into memory.

Set `show_zero_contours = True` to draw black **0 m/s contours** on both vertical sections, or `False` to hide them. Contours update with every frame and separate positive from negative flow. No line is drawn when a section has no sign change; masked land/bottom cells are excluded.

In [ ]:
import importlib.util
from IPython.display import HTML, display

analysis_directory = root / 'analysis'
def load_analysis_module(name):
    spec = importlib.util.spec_from_file_location(
        f'eddy_exp1_layer_{name}', analysis_directory / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

velocity_helpers = load_analysis_module('eddy_animation')
prepare_velocity_panels = velocity_helpers.prepare_velocity_panels
create_velocity_animation = velocity_helpers.create_velocity_animation

frame_stride = 1   # 1 = every daily record; 2 = every second daily record
interval_ms = 250  # default playback interval (4 frames per second)
show_zero_contours = True  # Black 0 m/s contours on both sections; False to hide

if not {'uo', 'vo'}.issubset(daily.data_vars):
    raise ValueError('This animation requires daily z-level uo and vo in ocean_daily.nc.')
velocity_panels = prepare_velocity_panels(daily, static, frame_stride=frame_stride)
print('Animated records:', velocity_panels['record_count'])
print('Fixed cuts (m):', velocity_panels['centre_x'], velocity_panels['centre_y'])
figure, eddy_animation = create_velocity_animation(
    velocity_panels, experiment_name=experiment_name,
    interval_ms=interval_ms, maximum_depth_m=4000,
    show_zero_contours=show_zero_contours,
)
# Inline JavaScript playback requires no ffmpeg or external video file.
with plt.rc_context({'animation.embed_limit': 40}):
    animation_html = eddy_animation.to_jshtml(fps=1000 / interval_ms, default_mode='loop')
plt.close(figure)
display(HTML(animation_html))

## Save the animation (optional)

Set `save_animation = True` below and run this cell **after the animation cell**. It saves an HTML player beside this notebook, with embedded frames and offline playback controls. The filename includes the experiment name and output segment; saving the same selection again replaces its previous export. Generated animation HTML files are ignored by Git. Leave the switch `False` to display the animation only.

In [ ]:
save_animation = False  # True = save beside this notebook; False = no file written

if save_animation:
    save_animation_html = load_analysis_module('animation_export').save_animation_html
    if 'animation_html' not in globals():
        raise RuntimeError('Run the animation cell before saving it.')
    animation_path = save_animation_html(
        animation_html, analysis_directory, experiment_name, output.name,
    )
    print('Saved animation:', animation_path)


## Online-remapped momentum diagnostics

Choose the daily record and a requested nominal depth. The nearest available
z-star centre is printed. These panels show **accelerations**, not their curls.
The primary curl budget below is computed at fixed native layer index.


In [ ]:
day_index = -1
requested_depth_m = 105.62
selected_z = budget_z.sel(z_l=requested_depth_m, method='nearest').isel(time=day_index)
print('Day:', selected_z.time.values, '| nominal centre (m):', float(selected_z.z_l))
remapped_fields = ['CAu', 'CAv', 'PFu', 'PFv', 'diffu', 'diffv']
limit = max(float(abs(selected_z[name]).max()) for name in remapped_fields)
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
for ax, name in zip(axes.flat, remapped_fields):
    selected_z[name].plot(ax=ax, cmap='RdBu_r', vmin=-limit, vmax=limit,
                          cbar_kwargs={'label': 'Acceleration (m s$^{-2}$)'})
    ax.set_title(f'{name}: nominal depth {float(selected_z.z_l):g} m')
    ax.set_aspect('equal')
plt.show()


## Native-layer relative and potential vorticity

PV is MOM6 layer PV, approximately `(f + RV) / h`, in m⁻¹ s⁻¹. Both fields
are at horizontal corners and cannot be remapped online in this MOM6 version.
`native_layer_index` is zero-based; layers have moving interface elevations,
not fixed geometric depths.


In [ ]:
native_layer_index = 0
selected_native = native.isel(time=day_index, zl=native_layer_index)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, name in zip(axes, ['RV', 'PV']):
    field = selected_native[name]
    limit = float(abs(field).max())
    field.plot(ax=ax, x='xq', y='yq', cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(f'{name}: native layer {native_layer_index + 1}')
    ax.set_aspect('equal')
plt.show()


## Native-layer momentum-curl budget

Loads the computed budget for this selected archive segment. Set
`recompute_budget=True` to regenerate it. Boundary corners and corners with any
adjacent layer thinner than 1 mm are excluded. The six independent terms are
summed; relative-vorticity and KE-gradient components are not added again.
Vertical friction includes bottom drag. PV products use daily mean fields.


In [ ]:
budget_helpers = load_analysis_module('layer_vorticity_budget')
budget_directory = root / 'analysis' / 'budget'
budget_path = budget_directory / 'layer_vorticity_budget.nc'
recompute_budget = False
if recompute_budget or not budget_path.exists():
    budget_helpers.calculate(output, budget_directory)
layer_budget = xr.open_dataset(budget_path, decode_times=time_decoder)
if Path(layer_budget.attrs['source_run']).resolve() != output.resolve():
    layer_budget.close()
    raise ValueError('Saved budget belongs to another segment. Set recompute_budget=True.')
datasets.append(layer_budget)
terms = budget_helpers.INDEPENDENT + ['net_tendency', 'residual']
selected = layer_budget.isel(time=day_index, zl=native_layer_index)
limit = max(float(abs(selected[name]).max()) for name in terms)
fig, axes = plt.subplots(2, 4, figsize=(17, 8), constrained_layout=True)
for ax, name in zip(axes.flat, terms):
    selected[name].plot(ax=ax, x='xq', y='yq', cmap='RdBu_r',
                       vmin=-limit, vmax=limit, add_colorbar=False)
    ax.set_title(name)
    ax.set_aspect('equal')
fig.colorbar(axes.flat[0].collections[0], ax=list(axes.flat), label='s$^{-2}$')
fig.suptitle(f'Native layer {native_layer_index + 1}: {selected.time.values}')
plt.show()
rms = lambda a: float(np.sqrt((a*a).mean(skipna=True)))
print('Selected layer/day RMS residual / net tendency:',
      rms(selected.residual) / rms(selected.net_tendency))
print('Whole-run RMS residual / net tendency:',
      rms(layer_budget.residual) / rms(layer_budget.net_tendency))


In [ ]:
from IPython.display import Image
print('Whole-run contribution magnitudes:')
display(Image(filename=str(budget_directory / 'budget_magnitudes.png')))


In [ ]:
for dataset in datasets:
    dataset.close()